# 2024 유로 스페인 빌드업 패턴: 좌우 비대칭 통계 검증

패스 네트워크의 "구조 유지" 판단과 구역 기반 전진 경로의 "왼쪽 쏠림" 판단이 지금까지 7장 이미지 정성 비교에 그쳤던 것을, 매치별 좌/우 비율을 계산해 paired t-test로 검증합니다(분석 질문 5).

- 세 지표 모두 "왼쪽 비율" = 왼쪽 / (왼쪽 + 오른쪽)로 정의하고, 완전 대칭(0.5)과의 차이를 `scipy.stats.ttest_1samp`(표본 7개, mu=0.5)로 검정합니다.
- 이 폴더(`asymmetry_stats/`)는 "2024 유로 스페인의 빌드업 패턴" 주제의 좌우 비대칭 통계 검증 방법론 전용 하위 폴더입니다. 분석 기획은 [`../PLAN.md`](../PLAN.md), 구역 기반 전진 경로는 [`../zone_progression/`](../zone_progression/), 패스 네트워크는 [`../pass_network/`](../pass_network/), 백로그 항목은 `ideas/backlog.md`의 "2024 유로 스페인의 빌드업 패턴"을 참고하세요.

## 방법론: 세 가지 좌/우 비율 지표

1. **구역 점유(전체 구역) 왼쪽 비율**: `zone_progression`과 동일한 30구역 그리드에서, 성공 패스 시작 위치가 왼쪽 채널(Left Wide/HS)인지 오른쪽 채널(Right HS/Wide)인지 집계해 왼쪽 / (왼쪽+오른쪽)을 계산합니다. 구역 기반 전진 경로의 "왼쪽 쏠림" 관찰을 전체 필드 기준으로 검증합니다.
2. **구역 점유(Att-Mid만) 왼쪽 비율**: 위와 같은 계산을 `Att-Mid` 가로단(하프라인을 갓 넘긴 구역 - 기존 `zone_progression/RESULTS.md`에서 "왼쪽 쏠림"의 실제 근거가 된 `Att-Mid/Left Wide` 구역이 속한 대)으로만 좁혀 계산합니다. 표본은 더 작지만(경기당 26~142개), 실제 관찰된 편중이 일어난 구역에 더 가깝습니다.
3. **패스 네트워크(4개 역할군) 왼쪽 비율**: `position` 라벨을 좌/우로 나눠, 7경기 전부에 공통으로 존재하는 4개 역할군(Back/Center Back/Defensive Midfield/Wing)의 슬롯별 패스 시도 수를 합산해 왼쪽 / (왼쪽+오른쪽)을 계산합니다. 목적지(구역)가 아니라 "선수가 얼마나 자주 볼을 만졌는가"(관여도)를 봅니다 - 패스 네트워크의 "구조 유지"(좌우 대칭적인 대형) 관찰을 검증합니다. 조지아전·프랑스전(5쌍)과 독일전(7쌍, 연장)은 역할군 라벨 집합이 더 크지만, 7경기 전부에 공통으로 존재하는 4개 역할군만 사용해 매치 간 비교가 항상 성립하도록 했습니다.

**데이터 검토** (`scripts/review_asymmetry_stats_data.py`, 7경기 전체): 구역 기준 좌/우 표본은 전체 구역 132~370개, Att-Mid만 26~142개로 검정에 충분했고, position 좌/우 라벨은 7경기 전부에서 완벽히 대응 쌍을 이뤘습니다(Left Back ↔ Right Back 등).

**한계**: 표본이 매치 7개뿐이라 paired t-test의 정규성 가정을 엄밀히 검증하기 어렵습니다 - 유의성 판단은 참고용으로, 평균 비율의 크기(효과 크기)와 함께 해석해야 합니다.

In [ ]:
import os
import sys

if sys.platform.startswith('win') and hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from mplsoccer import Pitch

from src.data_loader import get_competition_matches, get_match_events

COMPETITION_ID = 55  # UEFA Euro
SEASON_ID = 282      # 2024
TEAM = "Spain"

pitch = Pitch(pitch_type='statsbomb', positional=True)
x_edges = pitch.dim.positional_x
y_edges = pitch.dim.positional_y
ATT_MID_XI = 3

LEFT_Y = {0, 1}
RIGHT_Y = {3, 4}
COMMON_ROLE_STEMS = ['Back', 'Center Back', 'Defensive Midfield', 'Wing']

output_dir = os.path.join(os.getcwd(), "processed")
os.makedirs(output_dir, exist_ok=True)


def zone_index(x, y):
    xi = np.clip(np.searchsorted(x_edges, x, side='right') - 1, 0, len(x_edges) - 2)
    yi = np.clip(np.searchsorted(y_edges, y, side='right') - 1, 0, len(y_edges) - 2)
    return xi, yi


def match_zone_ratios(events):
    passes = events[(events['type'] == 'Pass') & (events['team'] == TEAM)].copy()
    passes = passes[passes['pass_outcome'].isna() & passes['pass_recipient'].notna()]
    passes['x'] = passes['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else np.nan)
    passes['y'] = passes['location'].apply(lambda loc: loc[1] if isinstance(loc, list) else np.nan)
    passes = passes.dropna(subset=['x', 'y'])

    zi = passes.apply(lambda r: zone_index(r['x'], r['y']), axis=1, result_type='expand')
    passes['xi'], passes['yi'] = zi[0], zi[1]

    left_all = (passes['yi'].isin(LEFT_Y)).sum()
    right_all = (passes['yi'].isin(RIGHT_Y)).sum()

    att_mid = passes[passes['xi'] == ATT_MID_XI]
    left_am = (att_mid['yi'].isin(LEFT_Y)).sum()
    right_am = (att_mid['yi'].isin(RIGHT_Y)).sum()

    return {
        'zone_all_left_ratio': left_all / (left_all + right_all),
        'zone_att_mid_left_ratio': left_am / (left_am + right_am),
    }


def match_network_ratio(events):
    team_events = events[events['team'] == TEAM]
    player_position = (
        team_events.dropna(subset=['position'])
        .groupby('player')['position']
        .agg(lambda s: s.value_counts().idxmax())
    )

    passes = events[(events['type'] == 'Pass') & (events['team'] == TEAM)].copy()
    passes = passes[passes['pass_outcome'].isna() & passes['pass_recipient'].notna()]
    passes['passer_position'] = passes['player'].map(player_position)

    slot_pass_count = passes.dropna(subset=['passer_position']).groupby('passer_position').size()

    left_total = sum(slot_pass_count.get(f'Left {stem}', 0) for stem in COMMON_ROLE_STEMS)
    right_total = sum(slot_pass_count.get(f'Right {stem}', 0) for stem in COMMON_ROLE_STEMS)
    return left_total / (left_total + right_total)

In [ ]:
matches = get_competition_matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
spain_matches = matches[(matches['home_team'] == TEAM) | (matches['away_team'] == TEAM)].copy()
spain_matches = spain_matches.sort_values('match_date')

rows = []
for _, match in spain_matches.iterrows():
    match_id = match['match_id']
    opponent = match['away_team'] if match['home_team'] == TEAM else match['home_team']
    stage = match['competition_stage']

    events = get_match_events(match_id=match_id)
    zone_ratios = match_zone_ratios(events)
    network_ratio = match_network_ratio(events)

    rows.append({
        'match': f"{stage} vs {opponent}",
        **zone_ratios,
        'network_left_ratio': network_ratio,
    })

result = pd.DataFrame(rows)
result

In [ ]:
metrics = [
    ('zone_all_left_ratio', '#00f0ff', 'Zone occupancy (all zones)'),
    ('zone_att_mid_left_ratio', '#ffe14d', 'Zone occupancy (Att-Mid only)'),
    ('network_left_ratio', '#ff5cad', 'Pass network (4 role pairs)'),
]

ttest_rows = []
for col, _, label in metrics:
    t_stat, p_value = stats.ttest_1samp(result[col], popmean=0.5)
    ttest_rows.append({
        'metric': label, 'mean_left_ratio': round(result[col].mean(), 3),
        'sd': round(result[col].std(), 3), 't_stat': round(t_stat, 3), 'p_value': round(p_value, 4),
    })

ttest_df = pd.DataFrame(ttest_rows)
ttest_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
fig.set_facecolor('#1e1e1e')
ax.set_facecolor('#1e1e1e')

y_positions = np.arange(len(result))
for i, (col, color, label) in enumerate(metrics):
    offset = (i - 1) * 0.22
    ax.scatter(result[col], y_positions + offset, color=color, s=70, label=label, zorder=3)

ax.axvline(0.5, color='white', linestyle='--', linewidth=1, alpha=0.7, zorder=1)
ax.set_yticks(y_positions)
ax.set_yticklabels(result['match'], color='white', fontsize=9)
ax.set_xlabel('Left ratio (0.5 = symmetric)', color='white', fontsize=10)
ax.tick_params(axis='x', colors='white')
ax.invert_yaxis()
for spine in ax.spines.values():
    spine.set_color('#555555')
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), facecolor='#1e1e1e', edgecolor='#555555',
          labelcolor='white', fontsize=9)
ax.set_title('Spain Left/Right Asymmetry by Match', color='white', fontsize=13, fontweight='bold', pad=14)
fig.tight_layout()

out_path = os.path.join(output_dir, 'spain_euro2024_asymmetry_stats.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#1e1e1e')
plt.show()
print('저장 완료:', out_path)

## 관찰 기록

세 지표의 통계 검정 결과와 해석을 정리한 결과는 `RESULTS.md`에 문서화할 예정입니다 (아직 미작성).